In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from line_profiler import LineProfiler, profile
from ipynb.fs.full.helpers import save_states, save_loglist
import time
import statistics
import os

In [2]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, obs):
        logits = self.net(obs)
        return Categorical(logits=logits)

In [3]:
class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)  # single scalar value
        )

    def forward(self, obs):
        # ensure output is always 1D tensor for advantage computation
        return self.net(obs).view(-1)

In [4]:
class ActorCriticAgent:
    def __init__(self, obs_dim, n_actions, lr=3e-4, gamma=0.99, device="cpu"):
        self.gamma = gamma
        self.device = device

        self.policy = PolicyNet(obs_dim, n_actions).to(device)
        self.value = ValueNet(obs_dim).to(device)

        self.optimizer = optim.Adam(list(self.policy.parameters()) + list(self.value.parameters()), lr=lr)

    @profile
    def run_episode(self, env, max_steps=100000, log_states=False, log_n_entries=400):
        #print("Started episode")
        log_probs = []
        values = []
        rewards = []
        dones = []
        episode_actions = []

        obs = env.reset()
        obs = torch.tensor(obs, dtype=torch.float32, device=self.device)

        states = []
        if log_states:
            step = max(1, math.floor(max_steps / log_n_entries))
        else : 
            step = 100

        for i in range(max_steps):
            dist = self.policy(obs)
            value = self.value(obs)
            action = dist.sample()
            log_prob = dist.log_prob(action)
            
            if log_states and i % step == 0:
                x, v, a, next_obs, reward, done = env.step(action.item())
                states.append({"m": env.m, "x": x, "v": v, "a": a, "t": env.t})
            else:
                _, _, _, next_obs, reward, done = env.step(action.item())

            episode_actions.append(action.item())
            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)
            dones.append(done)

            obs = torch.tensor(next_obs, dtype=torch.float32, device=self.device)

            if done:
                break

            # exceeded max episode steps
            if i >= max_steps:
                break

        # ensure obs has batch dimension for next_value
        obs_tensor = obs.unsqueeze(0) if obs.ndim == 1 else obs
        next_value = self.value(obs_tensor).detach().item() if not done else 0.0
            

        #print("Finished episode")
        if log_states:
            return log_probs, values, rewards, next_value, dones, episode_actions,  states
        else: 
            return log_probs, values, rewards, next_value, dones, episode_actions

    @profile
    def update(self, log_probs, values, rewards, next_value, dones):
        returns = []
        R = next_value
        
        for r, done in zip(reversed(rewards), reversed(dones)):
            R = r + self.gamma * R * (1.0 - float(done))  # cast done to float
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32, device=self.device)
        values = torch.stack(values)

        advantages = returns - values

        policy_loss = (-torch.stack(log_probs) * advantages.detach()).mean()
        value_loss = advantages.pow(2).mean()  # MSE
        loss = policy_loss + value_loss

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.item()

In [5]:
def train_ac(env, agent, n_episodes=1000, max_steps=100000, log_every=10, log_states=False, log_n_entries=400):
    states_list = []
    rewards_log = []
    actions_log = []
    elapsed_times = []
    # convergence 
    alpha = 0.1
    tol = 0.0001
    ema_reward = None

    folder = f"training_logs/mc_{n_episodes}_{max_steps}"
    episodes_folder = os.path.join(folder,"episodes")
    rewards_folder = os.path.join(folder,"rewards")
    actions_folder = os.path.join(folder,"actions")
    os.makedirs(episodes_folder, exist_ok=True)
    os.makedirs(rewards_folder, exist_ok=True)
    os.makedirs(actions_folder, exist_ok=True)

    for ep in range(1, n_episodes + 1):
        start = time.perf_counter()  # start timer

        if log_states and ep % log_every == 0:
            log_probs, values, rewards, next_value, dones, episode_actions, states = agent.run_episode(env, max_steps, log_states, log_n_entries)
        else:
            log_probs, values, rewards, next_value, dones, episode_actions, = agent.run_episode(env, max_steps)
        
        loss = agent.update(log_probs, values, rewards, next_value, dones)
        total_reward = sum(rewards)

        end = time.perf_counter()  # end timer
        elapsed = end - start
        elapsed_times.append(elapsed)

        if ep % log_every == 0:
            mean_elapsed = statistics.mean(elapsed_times)
            elapsed_times = []
            print(
                f"Episode {ep:4d} | "
                f"Return: {total_reward: .3e} | "
                f"Loss: {loss: .3e} | "
                f"Mean elapsed time per episode: {mean_elapsed:.3f} s"
            )

            if log_states:
                states_list.append(states)      
                rewards_log.append(total_reward)
                actions_log.append(episode_actions)
                save_states(os.path.join(episodes_folder, f"episode{ep}.json"), states)
                save_loglist(os.path.join(rewards_folder, f"rewards{ep}.json"), total_reward)
                save_loglist(os.path.join(actions_folder, f"actions{ep}.json"), episode_actions)

        # convergence check
        if ema_reward is None:
            ema_reward = total_reward
        else:
            prev_ema = ema_reward
            ema_reward = alpha * total_reward + (1 - alpha) * ema_reward

            if abs(ema_reward - prev_ema) < tol:
                print(f"Converged at episode {ep}")
                if log_states:
                    states_list.append(states)      
                    rewards_log.append(total_reward)
                    actions_log.append(episode_actions)
                    save_states(os.path.join(episodes_folder, f"episode{ep}.json"), states)
                    save_loglist(os.path.join(rewards_folder, f"rewards{ep}.json"), total_reward)
                    save_loglist(os.path.join(actions_folder, f"actions{ep}.json"), episode_actions)
                break

    return states_list, rewards_log, actions_log